# 02 — Augmentation Preview

Visualise the SimCLR augmentation pipeline applied to chest X-rays.
Shows side-by-side: original image | view 1 | view 2.

Goal: verify that augmentations are strong enough to be a useful pretext task
but not so aggressive that they destroy clinically relevant features.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import yaml
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from src.data.augmentations import SimCLRAugmentation

with open('../configs/pretrain_config.yaml') as f:
    config = yaml.safe_load(f)

augmentation = SimCLRAugmentation(config)
print('Augmentation pipeline loaded.')

In [ ]:
RAW_DIR = '../data/raw/nih-chest-xrays'
IMAGE_DIR = os.path.join(RAW_DIR, 'images')
PROCESSED_DIR = '../data/processed'

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, 'train.csv'))
sample_files = train_df['Image Index'].sample(6, random_state=7).tolist()
print(f'Selected {len(sample_files)} sample images.')

In [ ]:
def tensor_to_numpy(t):
    """Convert normalised tensor back to displayable uint8 image."""
    img = t.numpy().squeeze()  # (1, H, W) -> (H, W)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return (img * 255).astype(np.uint8)


fig, axes = plt.subplots(len(sample_files), 3, figsize=(9, 3 * len(sample_files)))

col_titles = ['Original', 'Augmented View 1', 'Augmented View 2']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight='bold')

for row, img_name in enumerate(sample_files):
    img_path = os.path.join(IMAGE_DIR, img_name)
    try:
        original = Image.open(img_path).convert('L').resize((224, 224))
        view1, view2 = augmentation(Image.open(img_path).convert('L'))

        axes[row, 0].imshow(np.array(original), cmap='gray')
        axes[row, 0].set_ylabel(img_name[:20], fontsize=7)
        axes[row, 1].imshow(tensor_to_numpy(view1), cmap='gray')
        axes[row, 2].imshow(tensor_to_numpy(view2), cmap='gray')
    except FileNotFoundError:
        for col in range(3):
            axes[row, col].text(0.5, 0.5, 'File not found',
                                ha='center', va='center', transform=axes[row, col].transAxes)

    for col in range(3):
        axes[row, col].axis('off')

plt.suptitle('SimCLR Augmentation Pairs for Chest X-rays', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Multiple augmentation samples for one image

Show 8 different views of the same X-ray to verify stochasticity.

In [ ]:
img_path = os.path.join(IMAGE_DIR, sample_files[0])
original = Image.open(img_path).convert('L')

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

axes[0].imshow(np.array(original.resize((224, 224))), cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

for i in range(1, 8):
    v1, _ = augmentation(original)
    axes[i].imshow(tensor_to_numpy(v1), cmap='gray')
    axes[i].set_title(f'View {i}')
    axes[i].axis('off')

plt.suptitle(f'8 random augmented views of: {sample_files[0][:30]}', fontsize=11)
plt.tight_layout()
plt.show()